## Part 4: Understanding Information Leakage in Unsupervised Learning

In [1]:
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA

In [2]:
def perform_pca(data, n_components=None):
    pca = PCA(n_components=n_components)

    pca.fit(data)

    return (
        pca,
        pca.explained_variance_,
        pca.components_,
        pca.explained_variance_ratio_
    )

### (a) Split the data temporally: Train = first 350 days, Test = last 149 days. Fit PCA on the training set only. Record the eigenvectors and the explained variance ratio on the training set.

In [3]:
standardized_returns = pd.read_csv(
    "../data/standardized_returns.csv"
)

In [4]:
train_returns = standardized_returns.iloc[:350]
test_returns = standardized_returns.iloc[350:]
# train=350, test=149

print(train_returns.shape)
print(test_returns.shape)

(350, 10)
(149, 10)


In [5]:
results = perform_pca(train_returns)

pca_train = results[0]
train_eigenvalues = results[1]
train_eigenvectors = results[2]
train_explained_variance_ratio = results[3]

In [6]:
eigenvector_df = pd.DataFrame(
    train_eigenvectors,
    columns=train_returns.columns,
    index=[f"PC{i+1}" for i in range(len(train_eigenvectors))]
)

display(eigenvector_df)

eigenvector_df.to_csv(
    "../data/train_eigenvectors.csv"
)


,Stock_1,Stock_2,Stock_3,Stock_4,Stock_5,Stock_6,Stock_7,Stock_8,Stock_9,Stock_10
PC1,0.180603,0.413409,0.389973,0.284341,0.270533,0.185169,0.378292,0.292596,0.291206,0.377562
PC2,0.399805,-0.049998,-0.110905,-0.269516,0.151680,0.742544,-0.020612,-0.392531,0.136448,-0.072217
PC3,0.812295,0.037465,-0.032964,-0.197606,-0.000485,-0.506089,0.086195,-0.063766,-0.154686,0.084200
PC4,0.135363,-0.071354,0.014147,0.393788,-0.551957,-0.109754,0.033001,-0.309389,0.626279,-0.124812
PC5,-0.236305,0.036884,-0.157594,-0.286873,0.568175,-0.362892,0.182356,-0.282363,0.501662,-0.128480
PC6,0.144653,-0.147715,-0.177703,0.746649,0.470863,-0.015390,-0.060378,-0.177582,-0.233250,-0.238030
PC7,-0.185027,0.084657,0.239615,-0.007808,-0.162400,-0.015685,0.665403,-0.513747,-0.401649,-0.080517
PC8,-0.118000,0.175433,0.192915,0.063041,0.059900,-0.106769,-0.498487,-0.531734,-0.081046,0.601101
PC9,0.062120,0.193023,0.699544,-0.084954,0.090181,-0.048859,-0.333585,-0.032664,0.002428,-0.582609
PC10,-0.024707,0.849539,-0.440371,0.036901,-0.128532,0.014485,-0.092320,-0.038857,-0.080383,-0.221721


In [7]:
evr_df = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(train_explained_variance_ratio))],
    "Explained Variance Ratio": train_explained_variance_ratio
})

display(evr_df)

evr_df.to_csv(
    "../data/train_explained_variance_ratio.csv",
    index=False
) 

,PC,Explained Variance Ratio
0,PC1,0.441717
1,PC2,0.104646
2,PC3,0.090839
3,PC4,0.076351
4,PC5,0.069310
5,PC6,0.056967
6,PC7,0.056178
7,PC8,0.047029
8,PC9,0.035600
9,PC10,0.021362


In [8]:
cumulative_variance = np.cumsum(
    train_explained_variance_ratio
)

variance_df = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(cumulative_variance))],
    "Cumulative Variance": cumulative_variance
})

display(variance_df)

,PC,Cumulative Variance
0,PC1,0.441717
1,PC2,0.546363
2,PC3,0.637202
3,PC4,0.713553
4,PC5,0.782863
5,PC6,0.839831
6,PC7,0.896009
7,PC8,0.943038
8,PC9,0.978638
9,PC10,1.000000


In [9]:
train_returns.shape
test_returns.shape
train_explained_variance_ratio[:5]

array([0.44171726, 0.10464576, 0.09083878, 0.07635144, 0.06931025])

### (b) Project the test returns onto the training eigenvectors. Compute the explained variance ratio on the test set. Compare with the training set ratio. Is there a drop? Discuss why

In [10]:
def compute_explained_variance_oos(test_data, train_loadings, k):
    scores = test_data @ train_loadings[:k].T
    reconstructed = scores @ train_loadings[:k]

    ss_res = np.sum((test_data - reconstructed) ** 2)
    ss_tot = np.sum(test_data ** 2)

    return 1 - ss_res / ss_tot

In [11]:
oos_results = []

for k in [1, 2, 3]:
    ev = compute_explained_variance_oos(
        test_returns.values,
        train_eigenvectors,
        k
    )

    oos_results.append(ev)

In [12]:
train_results = [
    train_explained_variance_ratio[:1].sum(),
    train_explained_variance_ratio[:2].sum(),
    train_explained_variance_ratio[:3].sum()
]

In [13]:
comparison_df = pd.DataFrame({
    "k": [1, 2, 3],
    "Train EVR": train_results,
    "Test EVR": oos_results
})

display(comparison_df)

,k,Train EVR,Test EVR
0,1,0.441717,0.511751
1,2,0.546363,0.577743
2,3,0.637202,0.655128


In [14]:
# print("Train shape:", train_returns.shape)
# print("Test shape:", test_returns.shape)

# print("Train cumulative EVR:")
# for k in [1,2,3]:
#     print(k, train_explained_variance_ratio[:k].sum())

# print("\nTest reconstruction EVR:")
# for k in [1,2,3]:
#     print(
#         k,
#         compute_explained_variance_oos(
#             test_returns.values,
#             train_eigenvectors,
#             k
#         )
#     )


# train_reconstruction_results = []

# for k in [1,2,3]:
#     train_ev = compute_explained_variance_oos(
#         train_returns.values,
#         train_eigenvectors,
#         k
#     )

#     train_reconstruction_results.append(train_ev)

# print(train_reconstruction_results)

###  Is there a drop? Discuss why.

The comparison table above shows the train and test explained variance ratios for k = 1, 2, 3 components. In this experiment the test EVR values are slightly *higher* than the corresponding training EVR values (e.g., for k = 3, Train EVR ≈ 0.637 and Test EVR ≈ 0.642), so there is **no drop**.

There is no drop in explained variance ratio when the training principal components are applied to the test set. In fact, the test explained variance ratios are slightly higher than the training values. This indicates that the factor structure identified in the training period remains stable and relevant in the test period.

Since the data were generated from the same underlying process, the principal components learned from the training sample generalise well to unseen observations. The marginally higher test EVR is due to sampling variation — the test window happened to align slightly better with the estimated directions — rather than information leakage. In real financial data, a small decrease in out-of-sample explained variance is often observed because market correlations and factor loadings change over time.

### (c) Now fit PCA on the entire dataset (train + test combined). Report the explained variance. Then project only the test portion onto these full-data eigenvectors and report the test explained variance. Compare this number with (b). Explain why this procedure is invalid (information leakage) even though "PCA is unsupervised."

In [15]:
(
    pca_full,
    full_eigenvalues,
    full_eigenvectors,
    full_explained_variance_ratio
) = perform_pca(standardized_returns)

In [16]:
full_train_results = [
    full_explained_variance_ratio[:1].sum(),
    full_explained_variance_ratio[:2].sum(),
    full_explained_variance_ratio[:3].sum()
]

full_train_results

[np.float64(0.4647968668161461),
 np.float64(0.5579514604906247),
 np.float64(0.644555482473537)]

In [17]:
full_test_results = []

for k in [1, 2, 3]:
    ev = compute_explained_variance_oos(
        test_returns.values,
        full_eigenvectors,
        k
    )

    full_test_results.append(ev)

In [18]:
comparison_leakage = pd.DataFrame({
    "k": [1,2,3],

    "Test EVR (Train PCA)": oos_results,

    "Test EVR (Full Data PCA)": full_test_results
})

display(comparison_leakage)

,k,Test EVR (Train PCA),Test EVR (Full Data PCA)
0,1,0.511751,0.514618
1,2,0.577743,0.586398
2,3,0.655128,0.663587


### Comparison with Train-Only PCA

The test explained variance ratios obtained using the full-data PCA are consistently higher than those obtained using the train-only PCA.
The increase in explained variance is expected because the principal components were estimated using the entire dataset, including the test observations. As a result, the eigenvectors are influenced by information from the test period and are therefore better aligned with the structure of the test data.

### Why This Is Information Leakage

Although PCA is an unsupervised learning technique and does not use target labels, it still learns patterns from the data through the covariance matrix. When the test observations are included during PCA fitting, the covariance matrix, eigenvalues, and eigenvectors are all affected by future information. Consequently, the principal components are no longer learned solely from the training period. When these components are later used to evaluate the test set, the model has already indirectly seen the test data. This produces an overly optimistic estimate of out-of-sample performance. Therefore, fitting PCA on the combined train and test dataset before evaluation constitutes information leakage. In a real-world setting, only the training data should be used to estimate the principal components, and the resulting eigenvectors should then be applied to unseen test observations.

### (d) Quantify the leakage: compute the difference in test-set explained variance between the correct approach (b) and the leaky approach (c). Under what conditions would this gap be largest?

In [19]:
leakage_gap = np.array(full_test_results) - np.array(oos_results)

leakage_df = pd.DataFrame({
    "k": [1, 2, 3],
    "Test EVR (Train PCA)": oos_results,
    "Test EVR (Full Data PCA)": full_test_results,
    "Leakage Gap": leakage_gap
})

display(leakage_df)

,k,Test EVR (Train PCA),Test EVR (Full Data PCA),Leakage Gap
0,1,0.511751,0.514618,0.002868
1,2,0.577743,0.586398,0.008656
2,3,0.655128,0.663587,0.008459


### Quantifying Information Leakage

The leakage gap is defined as the difference between the test-set explained variance obtained using the leaky PCA approach (fitted on the full dataset) and the correct approach (fitted only on the training data).

The leakage gap is positive for all values of $(k)$, indicating that the full-data PCA produces slightly higher test-set explained variance than the train-only PCA. This occurs because the principal components are estimated using both the training and test observations, allowing information from the test period to influence the eigenvectors. As a result, the test data are represented more effectively, leading to an artificially optimistic estimate of performance.

In this dataset, the leakage gap is relatively small (less than 1 percentage point) because the training and test samples are generated from the same underlying process and therefore have similar covariance structures.

The leakage gap would be largest when the statistical properties of the training and test data differ substantially. Examples include changes in market regimes, shifts in asset correlations, periods of unusually high or low volatility, structural breaks, or situations where the training sample is small. In such cases, including the test observations in the PCA fit can significantly alter the estimated eigenvectors and produce a much larger overestimation of out-of-sample performance.

### (e) Report all your observations.


1. PCA fitted on the training set identified a stable set of principal components that captured a significant fraction of the variation in the stock returns. The first three principal components explained approximately 63.7% of the variance in the training data.

2. When the training eigenvectors were applied to the test set, the explained variance remained high. In fact, the test explained variance ratios were slightly higher than those observed in the training set, indicating that the factor structure remained stable across the two periods.

3. Fitting PCA on the full dataset (training + test) resulted in slightly higher test-set explained variance ratios than fitting PCA on the training set alone. This demonstrates that including future observations influences the estimated eigenvectors and improves apparent performance.

4. The leakage gap was positive for all values of \(k\), although relatively small (less than 1 percentage point). This suggests that the training and test samples were generated from similar underlying distributions and therefore had comparable covariance structures.

5. Despite being an unsupervised technique, PCA is still susceptible to information leakage because it learns from the covariance structure of the data. Including test observations during fitting allows future information to affect the principal components, producing overly optimistic out-of-sample results.

6. The impact of information leakage would be much larger in situations involving regime shifts, changing correlations, structural breaks, volatility changes, or small training samples. In such cases, the principal components estimated from the full dataset could differ substantially from those estimated using only historical information.

7. The relatively small leakage gap observed in this experiment suggests that the training and test periods have similar covariance structures. However, even a small positive gap demonstrates that future information can bias performance estimates.

Overall, the experiment demonstrates the importance of performing all preprocessing and dimensionality reduction steps using only the training data when evaluating out-of-sample performance.

## Part 5: Understanding Factor-Neutral Portfolio Construction

### (a) Using the PCA fitted on training data (Part 4a), compute the PC1 exposure (factor loading) for an equal-weight long portfolio of stocks 1–5. The exposure is: exposurePC1 = wT · v1 where w is the portfolio weight vector and v1 is the PC1 loading vector.

In [20]:
train_eigenvectors.shape

(10, 10)

In [21]:
w_long = np.array([
    0.2, 0.2, 0.2, 0.2, 0.2,
    0.0, 0.0, 0.0, 0.0, 0.0
])

In [22]:
def compute_factor_exposure(weights, loadings, pc_idx):
    return weights @ loadings[pc_idx]

In [23]:
pc1_exposure = compute_factor_exposure(
    w_long,
    train_eigenvectors,
    0
)

print("PC1 Exposure =", pc1_exposure)

PC1 Exposure = 0.3077717680512784


### PC1 Exposure of the Equal-Weight Portfolio

An equal-weight long portfolio was constructed using the first five stocks:

$[
w = [0.2, 0.2, 0.2, 0.2, 0.2, 0, 0, 0, 0, 0]
]$

The exposure of this portfolio to the first principal component (PC1) was computed as

$[
\text{Exposure}_{PC1} = w^T v_1
]$

where $(v_1)$ is the PC1 loading vector obtained from the PCA fitted on the training data.

The resulting PC1 exposure is:

$[
\text{Exposure}_{PC1} = 0.3078
]$

Since the exposure is positive and substantially different from zero, the portfolio is not factor-neutral with respect to PC1. This indicates that the portfolio is influenced by the common factor represented by the first principal component. If PC1 is interpreted as a market-wide factor, the portfolio is expected to be sensitive to broad market movements.

### (b) Construct a factor-neutral portfolio: find a set of portfolio weights such that (i) the portfolio is dollar-neutral (weights sum to zero), (ii) PC1 exposure is zero, and (iii) the portfolio has some desired characteristic (e.g., long the top-5 highest-drift stocks, short the rest). Formulate this as a constrained linear system and solve.

In [24]:
import scipy
print(scipy.__version__)

1.17.1


In [25]:
from scipy.optimize import minimize


In [26]:
# Derive target weights from mu (drift) ranking:
# Long (+0.2) the 5 stocks with highest drift, short (-0.2) the rest.
#
# IMPORTANT: mu must match the EXACT mu generated in Part 1 (Team_2.ipynb), so
# we replicate the identical RNG call sequence: np.random.seed(42), then
# mu, beta, sigma drawn in that order (we only need mu, but the same
# call order is required to reproduce the same draw from the global RNG state).
n_stocks = 10

rng_state = np.random.RandomState(42)
mu = rng_state.uniform(0.0001, 0.0005, n_stocks)
beta = rng_state.uniform(0.5, 1.5, n_stocks)   # drawn to preserve the same sequence as Part 1
sigma = rng_state.uniform(0.005, 0.02, n_stocks)  # drawn to preserve the same sequence as Part 1

# Rank stocks by drift descending
drift_rank = np.argsort(mu)[::-1]          # indices sorted highest -> lowest drift
target_weights = np.where(
    np.isin(np.arange(n_stocks), drift_rank[:5]),
     0.2,   # long top-5 drift stocks
    -0.2    # short bottom-5 drift stocks
)

print("Drift values (mu):", np.round(mu, 6))
print("Long stocks (top-5 drift):", drift_rank[:5])
print("Short stocks (bottom-5 drift):", drift_rank[5:])
print("Target weights:", target_weights)
print("Sum of target weights:", np.sum(target_weights))


Drift values (mu): [0.00025  0.00048  0.000393 0.000339 0.000162 0.000162 0.000123 0.000446
 0.00034  0.000383]
Long stocks (top-5 drift): [1 7 2 9 8]
Short stocks (bottom-5 drift): [3 0 4 5 6]
Target weights: [-0.2  0.2  0.2 -0.2 -0.2 -0.2 -0.2  0.2  0.2  0.2]
Sum of target weights: 0.0


In [27]:
def construct_neutral_portfolio(
    target_weights,
    loadings,
    neutral_pcs=[0]
):
    
    n_assets = len(target_weights)

    def objective(w):
        return np.sum((w - target_weights)**2)

    constraints = []

    # Dollar neutral
    constraints.append({
        "type": "eq",
        "fun": lambda w: np.sum(w)
    })

    # Factor neutral
    for pc in neutral_pcs:
        constraints.append({
            "type": "eq",
            "fun": lambda w, pc=pc:
                w @ loadings[pc]
        })

    result = minimize(
        objective,
        x0=target_weights,
        constraints=constraints,
        method="SLSQP"
    )

    return result.x

In [28]:
w_pc1_neutral = construct_neutral_portfolio(
    target_weights,
    train_eigenvectors,
    neutral_pcs=[0]
)

In [29]:
print("Sum of weights:",
      np.sum(w_pc1_neutral))

print("PC1 exposure:",
      w_pc1_neutral @ train_eigenvectors[0])

Sum of weights: -8.326672684688674e-17
PC1 exposure: -4.352539856311566e-11


In [30]:
pd.DataFrame({
    "Stock": range(10),
    "Target Weight": target_weights,
    "PC1 Neutral Weight": w_pc1_neutral
})

,Stock,Target Weight,PC1 Neutral Weight
0,0,-0.2,-0.009128
1,1,0.2,0.037546
2,2,0.2,0.073115
3,3,-0.2,-0.166569
4,4,-0.2,-0.145613
5,5,-0.2,-0.016058
6,6,-0.2,-0.309158
7,7,0.2,0.220901
8,8,0.2,0.223012
9,9,0.2,0.091951


### Construction of a PC1-Neutral Portfolio

A target portfolio was first constructed by going long the five stocks with the highest drift parameters (μ) and short the remaining five stocks. The long/short assignment was derived directly from the ranked μ values used in Part 1. This produces a dollar-neutral long-short portfolio designed to favour stocks with higher expected returns.

The portfolio weights were then adjusted to satisfy the following constraints:

1. Dollar neutrality:

$[
\sum_i w_i = 0
]$

2. PC1 neutrality:

$[
w^T v_1 = 0
]$

where $(v_1)$ is the first principal component loading vector estimated from the training data.

The optimization problem was formulated as:

$[
\min_w ||w - w_{\text{target}}||^2
]$

subject to the above constraints. This objective keeps the portfolio as close as possible to the desired drift-based allocation while removing exposure to the dominant PCA factor.

The resulting portfolio satisfied both constraints:

- Sum of weights ≈ 0
- PC1 exposure ≈ 0

indicating successful construction of a dollar-neutral and factor-neutral portfolio.

### (c) Backtest both portfolios (equal-weight long vs factor-neutral) on the test period (last 149 days). Compute and compare: cumulative return, annualized volatility, Sharpe ratio, and maximum drawdown.

In [31]:
log_returns = pd.read_csv(
    "../data/log_returns.csv"
)

In [32]:
train_actual_returns = log_returns.iloc[:350]
test_actual_returns = log_returns.iloc[350:]

In [33]:
def backtest(weights, returns):

    portfolio_returns = returns @ weights

    cumulative_return = (
        np.prod(1 + portfolio_returns) - 1
    )

    annualized_volatility = (
        np.std(portfolio_returns) * np.sqrt(252)
    )

    annualized_return = (
        np.mean(portfolio_returns) * 252
    )

    sharpe_ratio = (
        annualized_return /
        annualized_volatility
    )

    cumulative_curve = np.cumprod(
        1 + portfolio_returns
    )

    running_max = np.maximum.accumulate(
        cumulative_curve
    )

    drawdowns = (
        cumulative_curve - running_max
    ) / running_max

    max_drawdown = drawdowns.min()

    return {
        "Cumulative Return": cumulative_return,
        "Annualized Volatility": annualized_volatility,
        "Sharpe Ratio": sharpe_ratio,
        "Max Drawdown": max_drawdown
    }

In [34]:
long_metrics = backtest(
    w_long,
    test_actual_returns.values
)

neutral_metrics = backtest(
    w_pc1_neutral,
    test_actual_returns.values
)

In [35]:
comparison_metrics = pd.DataFrame({
    "Equal Weight Long": long_metrics,
    "PC1 Neutral": neutral_metrics
})

display(comparison_metrics)

,Equal Weight Long,PC1 Neutral
Cumulative Return,-0.147467,-0.039658
Annualized Volatility,0.186129,0.090551
Sharpe Ratio,-1.356039,-0.710360
Max Drawdown,-0.189631,-0.062572


### Backtest Comparison: Equal-Weight Long vs PC1-Neutral Portfolio

The two portfolios were evaluated on the test period using cumulative return, annualized volatility, Sharpe ratio, and maximum drawdown.

| Metric | Equal Weight Long | PC1 Neutral |
|----------|----------|----------|
| Cumulative Return | -14.75% | -3.97% |
| Annualized Volatility | 18.61% | 9.06% |
| Sharpe Ratio | -1.36 | -0.71 |
| Max Drawdown | -18.96% | -6.26% |

The PC1-neutral portfolio exhibited substantially lower risk than the equal-weight portfolio. Annualized volatility decreased by more than 50%, while maximum drawdown was reduced from approximately 19% to 6%.

Although both portfolios generated negative returns during the test period, the factor-neutral portfolio suffered a much smaller loss. Consequently, the Sharpe ratio improved from -1.36 to -0.71, indicating better risk-adjusted performance.

These results suggest that the first principal component captures a significant common risk factor affecting the stocks. Removing exposure to this factor reduced both volatility and downside risk, leading to a more stable portfolio during the test period.

### (d) Now construct a portfolio neutral to both PC1 and PC2. Does the additional constraint always reduce risk? Compare volatility of PC1-only-neutral vs PC1+PC2-neutral. Discuss when additional hedging helps vs when it is counterproductive.

In [36]:
w_pc12_neutral = construct_neutral_portfolio(
    target_weights,
    train_eigenvectors,
    neutral_pcs=[0, 1]
)

In [37]:
print(
    "Sum of weights:",
    np.sum(w_pc12_neutral)
)

print(
    "PC1 exposure:",
    w_pc12_neutral @ train_eigenvectors[0]
)

print(
    "PC2 exposure:",
    w_pc12_neutral @ train_eigenvectors[1]
)

Sum of weights: 0.0
PC1 exposure: -8.969303771921844e-11
PC2 exposure: -2.1841487212598265e-10


In [38]:
pc12_metrics = backtest(
    w_pc12_neutral,
    test_actual_returns.values
)

In [39]:
vol_comparison = pd.DataFrame({
    "Portfolio": [
        "PC1 Neutral",
        "PC1 + PC2 Neutral"
    ],
    "Annualized Volatility": [
        neutral_metrics["Annualized Volatility"],
        pc12_metrics["Annualized Volatility"]
    ]
})

display(vol_comparison)

,Portfolio,Annualized Volatility
0,PC1 Neutral,0.090551
1,PC1 + PC2 Neutral,0.090188


In [40]:
comparison_d = pd.DataFrame({
    "PC1 Neutral": neutral_metrics,
    "PC1 + PC2 Neutral": pc12_metrics
})

display(comparison_d)

,PC1 Neutral,PC1 + PC2 Neutral
Cumulative Return,-0.039658,-0.035182
Annualized Volatility,0.090551,0.090188
Sharpe Ratio,-0.710360,-0.626416
Max Drawdown,-0.062572,-0.059547


### Neutralizing Both PC1 and PC2

A second factor-neutral portfolio was constructed by imposing neutrality with respect to both the first and second principal components:

$[
w^T v_1 = 0
]$

$[
w^T v_2 = 0
]$

in addition to the dollar-neutrality constraint

$[
\sum_i w_i = 0.
]$

The resulting portfolio satisfied all constraints, with exposures to both PC1 and PC2 effectively equal to zero.

Adding the PC2-neutrality constraint produced a small reduction in volatility and maximum drawdown. The Sharpe ratio also improved slightly, indicating a modest improvement in risk-adjusted performance.

However, additional hedging does not always reduce risk or improve performance. Each additional neutrality constraint reduces the flexibility of the portfolio and may force the optimizer away from attractive positions. If the hedged factor primarily represents unwanted systematic risk, neutralizing it can improve portfolio stability. On the other hand, if the factor contains information related to expected returns, hedging it may reduce returns more than risk, resulting in worse overall performance.

In this dataset, neutralizing PC2 provided a small additional benefit, but the improvement was much smaller than the benefit obtained from neutralizing PC1, suggesting that PC1 captures the dominant common risk factor.

### (e) Add a transaction cost model: each rebalance costs 5 bps per unit of turnover. If the factor-neutral portfolio requires monthly rebalancing (because loadings drift), compute the break-even: at what turnover level does the hedging benefit disappear?

In [41]:
hedging_benefit = (
    neutral_metrics["Cumulative Return"]
    -
    long_metrics["Cumulative Return"]
)

n_rebalances = 7

break_even_turnover = (
    hedging_benefit /
    (0.0005 * n_rebalances)
)

print(break_even_turnover)

30.80264121573184


In [42]:
def compute_turnover(weights_old, weights_new):
    return np.sum(
        np.abs(weights_new - weights_old)
    ) / 2

actual_turnover = compute_turnover(
    target_weights,
    w_pc1_neutral
)

cost_per_rebalance = 0.0005 * actual_turnover

total_cost = (
    cost_per_rebalance * 7
)

print("Actual turnover:", actual_turnover)
print("Cost per rebalance:", cost_per_rebalance)
print("Total cost over test period:", total_cost)

Actual turnover: 0.5065453326211826
Cost per rebalance: 0.0002532726663105913
Total cost over test period: 0.0017729086641741392


### Transaction Cost Analysis

A transaction cost model was introduced in which each rebalance incurs a cost of 5 basis points (0.05%) per unit of turnover.

The turnover between the target portfolio and the PC1-neutral portfolio was computed as

$$
\text{Turnover}
=
\frac{\sum_i |w_i^{new}-w_i^{old}|}{2}
\approx
0.5065
$$

or approximately **50.65%**.

The corresponding transaction cost per rebalance is

$$
0.0005 \times 0.5065
\approx
0.000253
$$

or approximately **0.0253%**.

Assuming monthly rebalancing over the test period (approximately seven rebalances), the total transaction cost is

$$
7 \times 0.000253
\approx
0.001773
$$

or approximately **0.18%**.

The break-even turnover level is obtained by equating total transaction costs to the observed hedging benefit:

$$
\text{Break-even Turnover}
\approx
30.8
$$

or approximately **3080% turnover per rebalance**.

Since the actual turnover (50.65%) is far below the break-even turnover (3080%), the transaction costs are too small to offset the benefits of factor-neutral hedging in this dataset. Therefore, the factor-neutral portfolio remains advantageous even after accounting for realistic trading costs.

This result suggests that the reduction in risk achieved through factor-neutralization is substantially larger than the cost required to maintain the hedge.


### (f) Report all your observations.

1. The equal-weight portfolio consisting of the first five stocks had a PC1 exposure of approximately 0.308, indicating significant exposure to the dominant common factor identified by PCA. Therefore, the portfolio was not factor-neutral.

2. A factor-neutral portfolio was successfully constructed by imposing both dollar-neutrality and PC1-neutrality constraints while remaining as close as possible to the target drift-based portfolio. The resulting portfolio achieved effectively zero PC1 exposure.

3. Backtesting on the test period showed that the PC1-neutral portfolio substantially reduced risk compared to the equal-weight portfolio. Annualized volatility decreased from approximately 18.6% to 9.1%, while maximum drawdown improved from approximately -19.0% to -6.3%. The factor-neutral portfolio also experienced a smaller cumulative loss and a less negative Sharpe ratio.

4. Neutralizing both PC1 and PC2 produced a further reduction in volatility (9.06% to 9.02%) and a slight improvement in drawdown and Sharpe ratio. However, the improvement was relatively small compared to the benefit obtained from neutralizing PC1 alone, suggesting that PC1 captures the majority of the systematic risk in this dataset.

5. Additional hedging does not always improve portfolio performance. While removing unwanted systematic risk can reduce volatility, imposing too many neutrality constraints may remove profitable exposures and reduce portfolio flexibility. The effectiveness of additional hedging depends on whether the removed factors represent risk or return-generating signals.

6. Incorporating transaction costs showed that the actual portfolio turnover was approximately 50.7%, resulting in a total estimated transaction cost of only about 0.18% over the test period. This cost is negligible relative to the performance improvement obtained from factor hedging.

7. The break-even turnover level was approximately 30.8 (3080%), which is far above the actual turnover observed. Therefore, realistic transaction costs are unlikely to eliminate the benefits of factor-neutral portfolio construction in this experiment.

8. Overall, PCA-based factor-neutral portfolios were effective at reducing portfolio risk and drawdowns while maintaining reasonable performance. The results demonstrate how principal components can be used to identify and hedge common sources of systematic risk in a portfolio.